# 04 · Model selection

**Question.** Which inputs, and which treatment of silence, without touching the test months?

**Answer.**
- Inputs: type of work, customer source, sending speed. Nothing else improves the score; gradient
  boosting is worse (log loss 0.709 vs 0.614).
- Silence: a decline-vs-quiet chance per customer gives the best score (0.5705) and the fewest
  miscalibrated groups (1).

**Protocol.** Rolling origins on 1 April, May, June and July; each fold learns from quotes settled by
its origin and is scored on that month. Quotes sent after 17 July are excluded so every label is in by
1 September. Criterion: log loss (lower is better); tie-break: groups miscalibrated beyond chance. AUC
is the chance a random yes is scored above a random no.

In [1]:
import sys

sys.path.insert(0, "../scripts")

import pandas as pd
import style

from quote_followup.config import load_settings
from quote_followup.data import load_quotes

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 100)
style.apply()
settings = load_settings("../config.toml")
quotes = load_quotes(settings.data_dir)

import ablations

candidates = ablations.with_candidate_inputs(quotes)

## 1. Inputs at sending (5,990 validation quotes)

In [2]:
ablations.send_time_ablations(candidates, settings).round(4)

,option,rows,auc,log_loss
0,"type of work, customer source, sending speed (chosen)",5990,0.6388,0.6137
1,"+ industry, customer type, quote size, number of items",5990,0.6373,0.6153
2,+ the business's past win rate,5990,0.6386,0.6136
3,+ the customer's past win rate,5990,0.6355,0.6164
4,+ sent on a weekend,5990,0.6386,0.6136
5,without type of work,5990,0.5812,0.6319
6,without customer source,5990,0.6234,0.6185
7,without sending speed (quote size instead),5990,0.6295,0.6165
8,gradient boosting with every fact,5990,0.5867,0.7086


All three inputs matter (type of work most). Extra inputs and boosting do not help.

## 2. Treatment of silence (24,151 quote-days)

In [3]:
ablations.silence_ablations(candidates, settings).round(4)

,option,rows,auc,log_loss,groups_missed_beyond_luck,worst_miss_z,luck_threshold_z
0,one says-no-or-goes-quiet figure per type of work,24151,0.7599,0.5717,3,4.6545,3.4101
1,says-no-or-goes-quiet chance per customer (chosen),24151,0.7616,0.5705,1,3.6614,3.4101
2,"same, reply speed also learned by customer source",24151,0.7612,0.5711,2,4.7092,3.4101
3,"same, reply speed also learned by sending speed",24151,0.7619,0.5704,3,4.7393,3.4101
4,"same, with gradient boosting instead",24151,0.7336,0.6137,7,4.8450,3.4101
5,days waiting as one more fact,24151,0.7592,0.5733,4,4.1777,3.4101
6,"days waiting as one more fact, gradient boosting",24151,0.7372,0.6257,3,3.8465,3.4101
7,days waiting only,24151,0.7190,0.5920,27,9.5049,3.4101


- Per-customer decline share beats one share per type of work (0.5705 vs 0.5717; 1 vs 3 groups).
- Reply speed by sending speed ties on log loss, but misses in more groups (3); by customer source it is worse.
- Adding days waiting as an input is worse (0.5733; 4 groups) and hides silence as a reason.
- Days waiting alone misses in 27 groups.

## 3. Chosen configuration

In [4]:
print(open("../config.toml").read().split("[followup]")[0])

# Every number that shapes a result lives here, so a run is fully described by this file and the data.

[data]
dir = "data"

[model]
# Chosen in notebooks/04 on folds that close before the backtest: industry, amount, line items,
# customer type and SP or customer history added nothing once these three were in. The same
# inputs feed both the win model and the decline-share model.
categorical = ["workflow_type", "customer_channel"]
numeric = ["log_hours_to_send"]
regularization_c = 1.0
# Train only on quotes sent at least this long before the scoring date. Quotes expire after
# 30 days and almost every decision lands inside 45, so their outcomes are settled and recent
# fast "yes" answers can't crowd out the slow "no" ones still pending.
label_horizon_days = 45
# Strata for the answer-time curves, coarsest first. Thin strata back off to coarser ones.
timing_segments = ["workflow_type"]
timing_min_quotes = 200


